# Lesson 9.2 - Advanced Computer Vision (detection/segmentation demo)

## Objectives

- Run pretrained detection and segmentation models from `torchvision`.
- Interpret outputs (boxes, labels, masks).
- Connect inference behavior to architecture choices.


In [ ]:
from __future__ import annotations

import torch
from PIL import Image, ImageDraw
import torchvision
from torchvision.transforms import functional as F
import matplotlib.pyplot as plt

print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


## Load Sample Image

To keep this notebook self-contained, we generate a simple synthetic image with geometric objects.


In [ ]:
img = Image.new("RGB", (400, 300), color=(235, 235, 235))
draw = ImageDraw.Draw(img)
draw.rectangle((50, 80, 170, 230), fill=(200, 70, 70))
draw.ellipse((220, 90, 360, 230), fill=(80, 120, 220))

plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.axis('off')
plt.title('Input Image')
plt.show()


## Object Detection with Faster R-CNN

Using pretrained COCO weights from `torchvision`.


In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

weights_det = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model_det = fasterrcnn_resnet50_fpn(weights=weights_det)
model_det.eval()

x = F.to_tensor(img)
with torch.no_grad():
    pred = model_det([x])[0]

print('keys:', pred.keys())
print('num detections:', len(pred['boxes']))


In [ ]:
score_thr = 0.4
labels_map = weights_det.meta["categories"]

for i in range(min(5, len(pred['boxes']))):
    score = float(pred['scores'][i])
    if score < score_thr:
        continue
    label_id = int(pred['labels'][i])
    label_name = labels_map[label_id] if label_id < len(labels_map) else str(label_id)
    box = pred['boxes'][i].tolist()
    print(f"det {i}: label={label_name}, score={score:.3f}, box={[round(v,1) for v in box]}")


## Semantic Segmentation with FCN-ResNet50


In [ ]:
from torchvision.models.segmentation import fcn_resnet50, FCN_ResNet50_Weights

weights_seg = FCN_ResNet50_Weights.DEFAULT
model_seg = fcn_resnet50(weights=weights_seg)
model_seg.eval()

x_seg = weights_seg.transforms()(img).unsqueeze(0)
with torch.no_grad():
    out = model_seg(x_seg)["out"]

mask = out.argmax(1).squeeze().cpu().numpy()
print('mask shape:', mask.shape, 'unique classes:', len(set(mask.flatten().tolist())))


In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(mask, cmap='tab20')
plt.axis('off')
plt.title('Segmentation Class Map')
plt.show()


## Connect to Theory

- Faster R-CNN (two-stage) emphasizes precise localization.
- Segmentation model produces dense per-pixel predictions.
- Task objective determines architecture and runtime characteristics.


## Business Case Studies & Exceptions

### Case A: Manufacturing Defect Inspection

Detection/segmentation models can localize defects for automated QA. Production success depends on camera consistency, labeling quality, and threshold tuning.

### Case B: Retail Shelf Analytics

Object detection tracks shelf availability and planogram compliance. Edge deployment may be required to reduce cloud costs and preserve latency.

### Exceptions

- For simple binary pass/fail checks, lightweight classifiers can outperform heavy detectors on cost and throughput.
- In privacy-sensitive environments, non-visual sensing may be preferable.


## Interview Questions & Answers

1. **Detection vs segmentation?**  
   Detection predicts boxes/classes; segmentation predicts pixel labels.
2. **When prefer two-stage detectors?**  
   When localization quality matters more than speed.
3. **What is semantic segmentation output?**  
   A class label for each pixel.
4. **What does `scores` represent in detection output?**  
   Confidence for each predicted box/class.
5. **How do you improve edge inference speed?**  
   Quantization, pruning, and optimized runtimes.
6. **Why do pretrained weights help?**  
   They transfer useful visual features to low-data tasks.
7. **How do you evaluate detection quality?**  
   mAP with IoU thresholds and per-class PR curves.
8. **What is a common CV deployment risk?**  
   Domain shift from training to production imagery.
9. **How do you reduce false positives?**  
   Threshold tuning, hard-negative mining, better calibration.
10. **Why monitor confidence distribution over time?**  
   It helps detect drift and model degradation.
